# Performer Attention Finetuning — Knowledge Distillation

**Goal:** Recover text generation quality after replacing softmax attention with FAVOR+ (Performer) in TinyLlama 1.1B.

**Method:** Knowledge distillation — the frozen softmax model (teacher) guides the performer model (student) to produce similar outputs, combined with standard language modeling loss.

**Loss:** `L = alpha * CE_loss + (1-alpha) * T^2 * KL_loss`
- CE = cross-entropy (predict next token correctly)
- KL = match the teacher's output distribution
- alpha=0.5, T=2.0

**Progressive strategy:** Start with 4/32 performer heads, gradually increase to 32/32.

**Requirements:** Any Colab GPU runtime. Go to `Runtime > Change runtime type > GPU`. No Triton needed — training uses pure PyTorch operations (autograd-compatible Python scan). Triton is only used at inference time for speed, and the code falls back automatically if unavailable.

## 0 — Setup

In [ ]:
!git clone https://github.com/Antoinechss/Performer-attention-LLM.git
# Pin transformers >= 4.40: required for the position_embeddings API used by MixedPerformerAttention
!pip install -q "transformers>=4.40.0" accelerate datasets sentencepiece protobuf

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# IMPORTANT: We do NOT cd into the repo — the local transformers/ folder
# would shadow the pip-installed HuggingFace transformers package.
# Instead we only add the performer/ directory to sys.path.
REPO_DIR = "/content/Performer-attention-LLM"
print(f"Repo cloned at: {REPO_DIR}")

In [ ]:
import sys, os, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Only add performer/ to path — NOT the repo root (which contains transformers/)
sys.path.insert(0, os.path.join(REPO_DIR, 'performer'))
from performer_attention import PerformerAttentionCore, _HAS_TRITON

from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from datasets import load_dataset

# ── Config ──────────────────────────────────────────────────────
MODEL_ID       = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DTYPE          = torch.float16
DEVICE         = "cuda"
SEQ_LEN        = 512
SEED           = 42

# Distillation hyperparameters
ALPHA          = 0.5    # weight for CE loss (1-alpha for KL)
TEMPERATURE    = 2.0    # softens teacher distributions

# Training hyperparameters
MICRO_BATCH    = 1      # batch=1 to fit teacher+student on T4 (15GB)
GRAD_ACCUM     = 8      # effective batch = 1 * 8 = 8
NUM_EPOCHS     = 2
SAVE_STEPS     = 500
EVAL_STEPS     = 250
MAX_TRAIN_SAMPLES = 20000  # subset of WikiText-103 for faster training

torch.manual_seed(SEED)
print(f"Config ready. Triton available: {_HAS_TRITON}")
print(f"HuggingFace transformers from: {AutoTokenizer.__module__}")

## 1 — MixedPerformerAttention (training-aware)

This class wraps a standard HuggingFace LlamaAttention layer, replacing some heads with FAVOR+ performer attention. Key differences from the analysis version:
- Omega is persistent (survives save/load)
- Triton is disabled during training (Python scan supports autograd)
- `num_performer_heads` can be changed between phases

In [ ]:
class MixedPerformerAttention(nn.Module):
    """Wraps HF LlamaAttention, routing configurable heads through FAVOR+."""

    def __init__(self, original_attn, num_performer_heads):
        super().__init__()
        self.head_dim = original_attn.head_dim
        self.num_heads = original_attn.config.num_attention_heads
        self.num_key_value_heads = original_attn.config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_key_value_heads
        self.scaling = self.head_dim ** -0.5
        self.num_performer_heads = num_performer_heads
        self.num_standard_heads = self.num_heads - num_performer_heads

        # Reuse pretrained projections (same nn.Linear objects, shared weights)
        self.q_proj = original_attn.q_proj
        self.k_proj = original_attn.k_proj
        self.v_proj = original_attn.v_proj
        self.o_proj = original_attn.o_proj

        # Performer core with persistent omega (fixed across training)
        self.performer_core = PerformerAttentionCore(
            head_dim=self.head_dim, num_features=256
        ).to(DEVICE)

        self.config = original_attn.config
        self.layer_idx = original_attn.layer_idx
        self.is_causal = True

    def _rotate_half(self, x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def _apply_rotary(self, q, k, cos, sin):
        cos = cos.unsqueeze(1)
        sin = sin.unsqueeze(1)
        return (q * cos) + (self._rotate_half(q) * sin), (k * cos) + (self._rotate_half(k) * sin)

    def forward(self, hidden_states, position_embeddings=None,
                attention_mask=None, past_key_values=None, **kwargs):
        B, N, _ = hidden_states.shape

        q = self.q_proj(hidden_states).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(B, N, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(B, N, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        cos, sin = position_embeddings
        q, k = self._apply_rotary(q, k, cos, sin)

        if past_key_values is not None:
            k, v = past_key_values.update(k, v, self.layer_idx)

        # Expand K/V for GQA
        if self.num_key_value_groups > 1:
            k = k.repeat_interleave(self.num_key_value_groups, dim=1)
            v = v.repeat_interleave(self.num_key_value_groups, dim=1)

        # Build causal mask when HF doesn't provide one.
        # The original LlamaSdpaAttention uses is_causal=True inside SDPA,
        # but our manual attention path needs an explicit mask.
        if attention_mask is None and N > 1 and self.num_standard_heads > 0:
            key_len = k.shape[2]
            causal = torch.full((N, key_len), torch.finfo(q.dtype).min,
                                device=q.device, dtype=q.dtype)
            attention_mask = torch.triu(causal, diagonal=key_len - N + 1)[None, None]

        if self.num_standard_heads == 0:
            attn_out = self.performer_core(q, k, v)
        elif self.num_performer_heads == 0:
            scores = torch.matmul(q, k.transpose(-2, -1)) * self.scaling
            if attention_mask is not None:
                scores = scores + attention_mask
            w = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)
            attn_out = torch.matmul(w, v)
        else:
            Kp = self.num_performer_heads
            out_p = self.performer_core(q[:, :Kp], k[:, :Kp], v[:, :Kp])
            q_s, k_s, v_s = q[:, Kp:], k[:, Kp:], v[:, Kp:]
            scores = torch.matmul(q_s, k_s.transpose(-2, -1)) * self.scaling
            if attention_mask is not None:
                scores = scores + attention_mask
            w = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_s.dtype)
            out_s = torch.matmul(w, v_s)
            attn_out = torch.cat([out_p, out_s], dim=1)

        attn_out = attn_out.transpose(1, 2).contiguous().reshape(B, N, -1)
        return self.o_proj(attn_out), None

print("MixedPerformerAttention defined.")

## 2 — Helper functions: patching, freezing, saving/loading

In [ ]:
def patch_model(model, num_performer_heads):
    """Replace all attention layers with MixedPerformerAttention."""
    for layer in model.model.layers:
        layer.self_attn = MixedPerformerAttention(
            layer.self_attn, num_performer_heads=num_performer_heads
        )
    return model


def set_performer_heads(model, num_performer_heads):
    """Change the number of performer heads without re-creating the model."""
    num_heads = model.config.num_attention_heads
    for layer in model.model.layers:
        layer.self_attn.num_performer_heads = num_performer_heads
        layer.self_attn.num_standard_heads = num_heads - num_performer_heads


def freeze_all(model):
    """Freeze all parameters."""
    for p in model.parameters():
        p.requires_grad = False


def unfreeze_qk(model):
    """Unfreeze Q and K projections in all layers."""
    for layer in model.model.layers:
        layer.self_attn.q_proj.weight.requires_grad = True
        layer.self_attn.k_proj.weight.requires_grad = True


def unfreeze_qkvo(model):
    """Unfreeze Q, K, V, O projections in all layers."""
    for layer in model.model.layers:
        for proj in [layer.self_attn.q_proj, layer.self_attn.k_proj,
                     layer.self_attn.v_proj, layer.self_attn.o_proj]:
            proj.weight.requires_grad = True


def count_params(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train


def save_checkpoint(model, optimizer, scheduler, step, phase, path):
    """Save training checkpoint including omega buffers."""
    # Collect omega matrices from all layers
    omegas = {}
    for i, layer in enumerate(model.model.layers):
        if hasattr(layer.self_attn, 'performer_core'):
            omegas[f"layer_{i}"] = layer.self_attn.performer_core.omega.cpu()

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'step': step,
        'phase': phase,
        'omegas': omegas,
    }, path)
    print(f"  Checkpoint saved: {path}")


def load_checkpoint(model, optimizer, scheduler, path):
    """Load training checkpoint and restore omega buffers."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)

    # Restore omegas first (before loading state_dict which may not include them)
    for i, layer in enumerate(model.model.layers):
        key = f"layer_{i}"
        if key in ckpt['omegas'] and hasattr(layer.self_attn, 'performer_core'):
            layer.self_attn.performer_core.omega.copy_(ckpt['omegas'][key].to(DEVICE))

    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f"  Checkpoint loaded: phase={ckpt['phase']}, step={ckpt['step']}")
    return ckpt['step'], ckpt['phase']


print("Helper functions defined.")

## 3 — Load models and dataset

In [ ]:
# ── Load teacher (frozen softmax model) ─────────────────────────
print("Loading teacher model (standard softmax)...")
teacher = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

# ── Load student (will be patched with performer) ──────────────
print("Loading student model...")
student = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE)

# Patch with performer attention (start with 4 heads for Phase 1)
INITIAL_PERFORMER_HEADS = 4
student = patch_model(student, num_performer_heads=INITIAL_PERFORMER_HEADS)

# Freeze everything, then unfreeze Q/K for Phase 1
freeze_all(student)
unfreeze_qk(student)

total, trainable = count_params(student)
print(f"\nStudent: {total/1e6:.0f}M total, {trainable/1e6:.0f}M trainable ({100*trainable/total:.1f}%)")
print(f"Performer heads: {INITIAL_PERFORMER_HEADS}/32")

# ── Tokenizer ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── VRAM check ─────────────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1e9
print(f"\nVRAM used: {allocated:.1f} GB")

In [ ]:
# ── Dataset: WikiText-103 ───────────────────────────────────────

class LMDataset(Dataset):
    """Pre-tokenized chunks of fixed length for causal LM training."""
    def __init__(self, token_ids, seq_len):
        # Trim to exact multiple of seq_len
        n = (len(token_ids) // seq_len) * seq_len
        self.data = token_ids[:n].view(-1, seq_len)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        return {"input_ids": x, "labels": x.clone()}


print("Loading WikiText-103...")
raw = load_dataset("wikitext", "wikitext-103-raw-v1")

# Tokenize entire training split, concatenate all tokens
print("Tokenizing...")
all_tokens = []
for text in raw["train"]["text"]:
    if text.strip():
        ids = tokenizer.encode(text, add_special_tokens=False)
        all_tokens.extend(ids)
all_tokens = torch.tensor(all_tokens, dtype=torch.long)
print(f"Total tokens: {len(all_tokens):,}")

# Create training dataset (subset for speed)
max_chunks = MAX_TRAIN_SAMPLES
train_ds = LMDataset(all_tokens, SEQ_LEN)
if len(train_ds) > max_chunks:
    train_ds.data = train_ds.data[:max_chunks]
print(f"Training samples: {len(train_ds)} chunks of {SEQ_LEN} tokens")

# Validation set (small, for eval)
val_tokens = []
for text in raw["validation"]["text"]:
    if text.strip():
        val_tokens.extend(tokenizer.encode(text, add_special_tokens=False))
val_tokens = torch.tensor(val_tokens, dtype=torch.long)
val_ds = LMDataset(val_tokens, SEQ_LEN)
if len(val_ds) > 200:
    val_ds.data = val_ds.data[:200]
print(f"Validation samples: {len(val_ds)} chunks")

train_loader = DataLoader(train_ds, batch_size=MICRO_BATCH, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=MICRO_BATCH, shuffle=False)

## 2b — Resume from checkpoint (optional)

If reconnecting after a Colab disconnect, run **cells 0–11** first (setup through dataset), then run this cell, then jump to the appropriate phase cell.

In [ ]:
# ── Resume from a previous Colab session ────────────────────────
# Edit RESUME_CHECKPOINT to the path of your checkpoint file.
# Then run cells 0–3 (setup, imports, MixedPerformerAttention, helpers),
# then run this cell, then jump straight to the phase you want to continue.

RESUME_CHECKPOINT = None  # e.g. "/content/drive/MyDrive/performer_checkpoints/best_phase1_K4_QK.pt"

if RESUME_CHECKPOINT is not None:
    from google.colab import drive
    drive.mount('/content/drive')

    # Determine which phase this checkpoint belongs to so we set the right heads
    # The phase name is stored in the checkpoint — we load it to find out.
    _meta = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=False)
    _phase = _meta['phase']
    _step  = _meta['step']
    print(f"Resuming from: phase={_phase}, step={_step}")

    # Map phase name → performer head count and unfreeze function
    _PHASE_CONFIG = {
        "phase1_K4_QK":    (4,  unfreeze_qk),
        "phase2_K8_QKVO":  (8,  unfreeze_qkvo),
        "phase3_K16_QKVO": (16, unfreeze_qkvo),
        "phase4_K32_QKVO": (32, unfreeze_qkvo),
    }
    _heads, _unfreeze_fn = _PHASE_CONFIG[_phase]

    # Re-create optimizer + scheduler (will be overwritten by load_checkpoint)
    set_performer_heads(student, _heads)
    freeze_all(student)
    _unfreeze_fn(student)
    _optimizer = torch.optim.AdamW(
        [p for p in student.parameters() if p.requires_grad],
        lr=2e-5, weight_decay=0.01
    )
    _total_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM
    _scheduler = get_cosine_schedule_with_warmup(_optimizer, max(_total_steps // 20, 10), _total_steps)

    load_checkpoint(student, _optimizer, _scheduler, RESUME_CHECKPOINT)
    print(f"Ready to continue training from step {_step}.")
    print("Now run the appropriate phase cell below (Phase 1/2/3/4).")
else:
    print("RESUME_CHECKPOINT is None — skipping. Set it to a .pt path to resume.")

## 4 — Training loop with distillation

The core training function. Handles:
- Mixed precision (fp16 forward, fp32 loss)
- Gradient accumulation
- CE + KL distillation loss
- Periodic evaluation and checkpointing

In [ ]:
@torch.no_grad()
def evaluate(student, teacher, val_loader):
    """Compute validation perplexity and KL divergence from teacher."""
    student.eval()
    total_ce, total_kl, n_tokens = 0.0, 0.0, 0

    for batch in val_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast('cuda', dtype=DTYPE):
            s_out = student(input_ids=input_ids, use_cache=False)
            t_out = teacher(input_ids=input_ids, use_cache=False)

        s_logits = s_out.logits[:, :-1].float()
        t_logits = t_out.logits[:, :-1].float()
        tgt = labels[:, 1:].contiguous()

        ce = F.cross_entropy(s_logits.reshape(-1, s_logits.size(-1)),
                             tgt.reshape(-1), reduction='sum')

        s_log_probs = F.log_softmax(s_logits / TEMPERATURE, dim=-1)
        t_probs = F.softmax(t_logits / TEMPERATURE, dim=-1)
        kl = F.kl_div(s_log_probs, t_probs, reduction='sum') * (TEMPERATURE ** 2)

        n = tgt.numel()
        total_ce += ce.item()
        total_kl += kl.item()
        n_tokens += n

    avg_ce = total_ce / n_tokens
    avg_kl = total_kl / n_tokens
    ppl = math.exp(min(avg_ce, 20))  # cap to avoid overflow
    student.train()
    return ppl, avg_kl


def train_phase(student, teacher, train_loader, val_loader,
                num_performer_heads, unfreeze_fn, lr, num_epochs, phase_name,
                checkpoint_dir="checkpoints"):
    """Run one training phase. No GradScaler — weights stay fp16, loss computed in fp32."""

    os.makedirs(checkpoint_dir, exist_ok=True)
    set_performer_heads(student, num_performer_heads)

    freeze_all(student)
    unfreeze_fn(student)
    total, trainable = count_params(student)

    print(f"\n{'='*70}")
    print(f"Phase: {phase_name}")
    print(f"Performer heads: {num_performer_heads}/32")
    print(f"Trainable: {trainable/1e6:.0f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%)")
    print(f"LR: {lr}, Epochs: {num_epochs}")
    print(f"{'='*70}\n")

    optimizer = torch.optim.AdamW(
        [p for p in student.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01
    )

    total_steps = len(train_loader) * num_epochs // GRAD_ACCUM
    warmup_steps = max(total_steps // 20, 10)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    student.train()
    global_step = 0
    log_interval = 50
    best_ppl = float('inf')

    for epoch in range(num_epochs):
        epoch_loss, epoch_ce, epoch_kl = 0.0, 0.0, 0.0
        t0 = time.time()

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            # Forward with autocast (fp16), loss in fp32
            with torch.amp.autocast('cuda', dtype=DTYPE):
                s_out = student(input_ids=input_ids, use_cache=False)
                with torch.no_grad():
                    t_out = teacher(input_ids=input_ids, use_cache=False)

            s_logits = s_out.logits[:, :-1].float()
            t_logits = t_out.logits[:, :-1].float()
            tgt = labels[:, 1:].contiguous()

            ce_loss = F.cross_entropy(
                s_logits.reshape(-1, s_logits.size(-1)), tgt.reshape(-1)
            )
            s_log_probs = F.log_softmax(s_logits / TEMPERATURE, dim=-1)
            t_probs = F.softmax(t_logits / TEMPERATURE, dim=-1)
            kl_loss = F.kl_div(s_log_probs, t_probs, reduction='batchmean') * (TEMPERATURE ** 2)

            loss = (ALPHA * ce_loss + (1 - ALPHA) * kl_loss) / GRAD_ACCUM
            loss.backward()

            epoch_ce += ce_loss.item()
            epoch_kl += kl_loss.item()
            epoch_loss += loss.item() * GRAD_ACCUM

            # Gradient accumulation step
            if (batch_idx + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in student.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

                if global_step % log_interval == 0:
                    avg_l = epoch_loss / (batch_idx + 1)
                    avg_ce = epoch_ce / (batch_idx + 1)
                    avg_kl = epoch_kl / (batch_idx + 1)
                    elapsed = time.time() - t0
                    lr_now = scheduler.get_last_lr()[0]
                    print(f"  step {global_step:>5d} | loss {avg_l:.4f} | "
                          f"CE {avg_ce:.4f} | KL {avg_kl:.4f} | "
                          f"lr {lr_now:.2e} | {elapsed:.0f}s")

                if global_step % EVAL_STEPS == 0:
                    ppl, val_kl = evaluate(student, teacher, val_loader)
                    print(f"  >> eval | ppl {ppl:.2f} | KL {val_kl:.4f}")
                    if ppl < best_ppl:
                        best_ppl = ppl
                        save_checkpoint(student, optimizer, scheduler,
                                        global_step, phase_name,
                                        os.path.join(checkpoint_dir, f"best_{phase_name}.pt"))
                    student.train()

                if global_step % SAVE_STEPS == 0:
                    save_checkpoint(student, optimizer, scheduler,
                                    global_step, phase_name,
                                    os.path.join(checkpoint_dir, f"{phase_name}_step{global_step}.pt"))

        # End of epoch
        ppl, val_kl = evaluate(student, teacher, val_loader)
        print(f"\n  Epoch {epoch+1} done | val ppl {ppl:.2f} | val KL {val_kl:.4f} | "
              f"best ppl {best_ppl:.2f}")
        if ppl < best_ppl:
            best_ppl = ppl
            save_checkpoint(student, optimizer, scheduler,
                            global_step, phase_name,
                            os.path.join(checkpoint_dir, f"best_{phase_name}.pt"))

    print(f"\n  Phase {phase_name} complete. Best ppl: {best_ppl:.2f}")
    return best_ppl


print("Training functions defined (no GradScaler, batch=1).")

## 5 — Run training phases

Each phase increases the number of performer heads. Run them sequentially. If a Colab session disconnects, re-run cells 0-4, then load the latest checkpoint and continue from the appropriate phase.

In [ ]:
# ── Baseline eval (before any training) ─────────────────────────
print("Baseline evaluation (no finetuning)...")
set_performer_heads(student, 4)
student.eval()
ppl_base, kl_base = evaluate(student, teacher, val_loader)
print(f"Baseline (4 performer heads, no training): ppl={ppl_base:.2f}, KL={kl_base:.4f}\n")

# Verify gradients flow
student.train()
test_batch = next(iter(train_loader))
test_ids = test_batch["input_ids"].to(DEVICE)
with torch.amp.autocast('cuda', dtype=DTYPE):
    out = student(input_ids=test_ids, use_cache=False)
loss = out.logits[:, :-1].float().sum()
loss.backward()

grad_ok = all(
    layer.self_attn.q_proj.weight.grad is not None
    for layer in student.model.layers
)
print(f"Gradient check (Q_proj): {'PASS' if grad_ok else 'FAIL'}")
student.zero_grad()
print("Ready for training.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1: 4/32 performer heads, train Q/K only
# ══════════════════════════════════════════════════════════════════
ppl_1 = train_phase(
    student, teacher, train_loader, val_loader,
    num_performer_heads=4,
    unfreeze_fn=unfreeze_qk,
    lr=2e-5,
    num_epochs=NUM_EPOCHS,
    phase_name="phase1_K4_QK",
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 2: 8/32 performer heads, train Q/K/V/O
# ══════════════════════════════════════════════════════════════════
ppl_2 = train_phase(
    student, teacher, train_loader, val_loader,
    num_performer_heads=8,
    unfreeze_fn=unfreeze_qkvo,
    lr=2e-5,
    num_epochs=NUM_EPOCHS,
    phase_name="phase2_K8_QKVO",
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 3: 16/32 performer heads, train Q/K/V/O
# ══════════════════════════════════════════════════════════════════
ppl_3 = train_phase(
    student, teacher, train_loader, val_loader,
    num_performer_heads=16,
    unfreeze_fn=unfreeze_qkvo,
    lr=1e-5,
    num_epochs=NUM_EPOCHS,
    phase_name="phase3_K16_QKVO",
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 4: 32/32 performer heads (fully performer), train Q/K/V/O
# ══════════════════════════════════════════════════════════════════
ppl_4 = train_phase(
    student, teacher, train_loader, val_loader,
    num_performer_heads=32,
    unfreeze_fn=unfreeze_qkvo,
    lr=1e-5,
    num_epochs=NUM_EPOCHS,
    phase_name="phase4_K32_QKVO",
)

## 6 — Evaluation: text generation + quality comparison

In [ ]:
# ── Summary ─────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("TRAINING SUMMARY")
print(f"{'='*70}")
print(f"  Baseline (4 heads, no training): ppl = {ppl_base:.2f}")
print(f"  Phase 1 (4 heads, Q/K):          ppl = {ppl_1:.2f}")
print(f"  Phase 2 (8 heads, Q/K/V/O):      ppl = {ppl_2:.2f}")
print(f"  Phase 3 (16 heads, Q/K/V/O):     ppl = {ppl_3:.2f}")
print(f"  Phase 4 (32 heads, Q/K/V/O):     ppl = {ppl_4:.2f}")
print(f"{'='*70}\n")

In [ ]:
# ── Text generation comparison ──────────────────────────────────
PROMPT = "<|user|>\nHow do I get a good night's sleep?</s>\n<|assistant|>\n"
GEN_TOKENS = 100

prompt_ids = tokenizer(PROMPT, return_tensors="pt")["input_ids"].to(DEVICE)

# Teacher (standard softmax)
with torch.no_grad():
    teacher_ids = teacher.generate(prompt_ids, max_new_tokens=GEN_TOKENS, do_sample=False)
teacher_text = tokenizer.decode(teacher_ids[0], skip_special_tokens=True)

# Student (finetuned performer, all 32 heads)
set_performer_heads(student, 32)
student.eval()
with torch.no_grad():
    cur = prompt_ids.clone()
    for _ in range(GEN_TOKENS):
        out = student(input_ids=cur, use_cache=False)
        next_id = out.logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        cur = torch.cat([cur, next_id], dim=-1)
        if next_id.item() == tokenizer.eos_token_id:
            break
student_text = tokenizer.decode(cur[0], skip_special_tokens=True)

print(f"{'='*70}")
print("TEXT GENERATION COMPARISON (after finetuning)")
print(f"{'='*70}\n")
print("--- Teacher (softmax) ---")
print(teacher_text)
print(f"\n--- Student (32/32 performer, finetuned) ---")
print(student_text)

In [ ]:
# ── Mixed-head quality sweep (Section C from analysis) ──────────
print(f"\n{'='*70}")
print("QUALITY SWEEP — KL divergence vs performer head count (after finetuning)")
print(f"{'='*70}\n")

student.eval()
with torch.no_grad():
    std_ref = teacher(input_ids=prompt_ids, use_cache=False)
std_logits_ref = std_ref.logits[0, -1].float()
std_probs_ref = F.softmax(std_logits_ref, dim=-1)
std_top5_ref = set(std_logits_ref.topk(5).indices.tolist())

num_heads = student.config.num_attention_heads
print(f"{'K':>4}  {'KL':>7}  {'Top5':>5}  {'p(top1)':>8}")
print("-" * 30)

for k in [0, 1, 2, 4, 8, 16, 32]:
    set_performer_heads(student, k)
    with torch.no_grad():
        out = student(input_ids=prompt_ids, use_cache=False)
    logits = out.logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    kl = F.kl_div(probs.log(), std_probs_ref, reduction='sum').item()
    overlap = len(set(logits.topk(5).indices.tolist()) & std_top5_ref)
    p_top1 = probs[std_logits_ref.argmax().item()].item()
    print(f"{k:>4}  {kl:>7.3f}  {overlap:>4}/5  {p_top1:>8.1%}")

## 7 — Save to Google Drive (optional)

Mount Google Drive and copy checkpoints for persistence across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/performer_checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy all checkpoints to Drive
import shutil
for f in os.listdir("checkpoints"):
    src = os.path.join("checkpoints", f)
    dst = os.path.join(DRIVE_DIR, f)
    shutil.copy2(src, dst)
    print(f"Copied {f} -> Drive")

print(f"\nCheckpoints saved to {DRIVE_DIR}")